# Plant Disease Detection
DenseNet121 fine-tuned on the PlantVillage dataset (15 classes).

**Kaggle dataset:** Add `abdallahalidev/plantvillage-dataset` (or `emmarex/plantdisease`) to the notebook inputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from keras.applications import DenseNet121
from keras.applications.densenet import preprocess_input
from keras.utils import load_img, img_to_array
from sklearn.metrics import confusion_matrix

print('TF version:', tf.__version__)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
IMAGE_SIZE  = 224
BATCH_SIZE  = 32
SEED        = 42
NUM_CLASSES = 15

# If using abdallahalidev/plantvillage-dataset, change to:
#   DATA_DIR = '../input/plantvillage-dataset/color/'
DATA_DIR = '../input/plantdisease/PlantVillage/'

DISEASE_CLASSES = [
    'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy',
    'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight',
    'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot',
    'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus',
    'Tomato_healthy'
]

In [ ]:
# ── Data exploration ─────────────────────────────────────────────────────────
class_counts = {
    cls: len(os.listdir(os.path.join(DATA_DIR, cls)))
    for cls in DISEASE_CLASSES
    if os.path.exists(os.path.join(DATA_DIR, cls))
}

plt.figure(figsize=(14, 5))
plt.bar(range(len(class_counts)), list(class_counts.values()), color='steelblue')
plt.xticks(range(len(class_counts)), list(class_counts.keys()), rotation=90)
plt.title('Images per Disease Class')
plt.tight_layout()
plt.show()
print('Total images:', sum(class_counts.values()))

In [ ]:
# Sample images — load_img uses PIL so always RGB, no BGR issue
sample_class = 'Tomato_Bacterial_spot'
sample_files = os.listdir(os.path.join(DATA_DIR, sample_class))[:9]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, fname in zip(axes.flatten(), sample_files):
    img = load_img(os.path.join(DATA_DIR, sample_class, fname), target_size=(224, 224))
    ax.imshow(img)
    ax.axis('off')
plt.suptitle(sample_class, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Data generators ──────────────────────────────────────────────────────────
# flow_from_directory loads batches on demand — no OOM at 224x224
# preprocess_input matches DenseNet121 ImageNet pretraining (mean subtraction)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=360,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    classes=DISEASE_CLASSES
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False,
    classes=DISEASE_CLASSES
)

print(f'Training samples  : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
def build_model():
    base = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
    base.trainable = False

    inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs, output), base

model, base_model = build_model()
model.summary()

In [ ]:
# ── Phase 1: frozen base, train top layers ───────────────────────────────────
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    ModelCheckpoint('model.h5', save_best_only=True, monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)
]

hist1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks
)
print('Phase 1 complete')

In [ ]:
# ── Phase 2: unfreeze top 50 DenseNet layers, fine-tune ─────────────────────
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks
)
print('Phase 2 complete')

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
final_loss, final_accuracy = model.evaluate(val_gen)
print(f'Final Loss: {final_loss:.4f}  |  Final Accuracy: {final_accuracy:.4f}')

Y_pred = np.argmax(model.predict(val_gen), axis=1)
Y_true = val_gen.classes

cm = confusion_matrix(Y_true, Y_pred)
plt.figure(figsize=(14, 14))
ax = sns.heatmap(cm, cmap='Greens', annot=True, square=True,
                 xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES)
ax.set_ylabel('Actual', fontsize=13)
ax.set_xlabel('Predicted', fontsize=13)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# ── Training history ─────────────────────────────────────────────────────────
acc      = hist1.history['accuracy']     + hist2.history['accuracy']
val_acc  = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
loss     = hist1.history['loss']         + hist2.history['loss']
val_loss = hist1.history['val_loss']     + hist2.history['val_loss']
split    = len(hist1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, train, val, title in [
    (ax1, acc,  val_acc,  'Accuracy'),
    (ax2, loss, val_loss, 'Loss')
]:
    ax.plot(train, label='train')
    ax.plot(val,   label='val')
    ax.axvline(split, color='red', linestyle='--', label='fine-tune start')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Single image inference test ──────────────────────────────────────────────
test_path = '../input/plantdisease/PlantVillage/Potato___Early_blight/042135e2-e126-4900-9212-d42d900b8125___RS_Early.B 8791.JPG'

img = load_img(test_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
plt.imshow(img)
plt.axis('off')
plt.show()

x = img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
ind   = np.argmax(preds[0])
print(f'Prediction : {DISEASE_CLASSES[ind]}')
print(f'Confidence : {preds[0][ind]*100:.1f}%')